# FinTrustX Model Evaluation

**Purpose:**  
Evaluate trained models on the held-out test set and identify the best-performing model for credit-risk prediction.

Because this dataset is highly imbalanced, relying solely on Accuracy is misleading. **ROC AUC** is the primary selection metric, while **Average Precision**, **Recall**, **Precision**, and **F1 Score** are also heavily considered to ensure the model successfully detects defaults without overly penalizing good customers.

In [ ]:
import sys
import os
from pathlib import Path
import json

print(f"Python version: {sys.version}")

try:
    import numpy as np
    print(f"NumPy version: {np.__version__}")
except ImportError:
    print("NumPy not installed")

try:
    import pandas as pd
    print(f"Pandas version: {pd.__version__}")
except ImportError:
    print("Pandas not installed")

try:
    import sklearn
    print(f"scikit-learn version: {sklearn.__version__}")
except ImportError:
    print("scikit-learn not installed")

try:
    import tensorflow as tf
    print(f"TensorFlow version: {tf.__version__}")
except ImportError:
    print("TensorFlow not installed")

try:
    import xgboost as xgb
    print(f"XGBoost version: {xgb.__version__}")
except ImportError:
    print("XGBoost not installed")

try:
    import catboost as cb
    print(f"CatBoost version: {cb.__version__}")
except ImportError:
    print("CatBoost not installed")

try:
    import optuna
    print(f"Optuna version: {optuna.__version__}")
except ImportError:
    print("Optuna not installed")

In [ ]:
# Resolve project root robustly
notebook_dir = Path.cwd()
if notebook_dir.name == 'notebooks':
    project_root = notebook_dir.parent
elif notebook_dir.name == 'Credit-risk-ai':
    project_root = notebook_dir
else:
    # Fallback if run from somewhere else
    project_root = Path(r'D:\Projects\Credit-risk-ai')

print(f"Project root resolved to: {project_root}")

required_dirs = ['data', 'models', 'reports', 'studies']
for d in required_dirs:
    d_path = project_root / d
    if d_path.exists():
        print(f"✅ Found directory: {d}")
    else:
        print(f"❌ Missing directory: {d}")

In [ ]:
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

try:
    from src.config import MODEL_DIR, REPORT_DIR, MODEL_NAMES, DEEP_LEARNING_TRAINING
    print("✅ Imported src.config successfully.")
except ImportError as e:
    print(f"❌ Error importing src.config: {e}")
    # Fallbacks in case config fails
    MODEL_DIR = project_root / "models"
    REPORT_DIR = project_root / "reports"
    MODEL_NAMES = {
        "decision_tree": "decision_tree.joblib",
        "random_forest": "random_forest.joblib",
        "xgboost": "xgboost.joblib",
        "catboost": "catboost.joblib",
        "neural_network": "neural_network.keras"
    }
    DEEP_LEARNING_TRAINING = {"target_column": "TARGET"}

try:
    from src.evaluate import evaluate_model, business_metrics
    print("✅ Imported src.evaluate successfully.")
except ImportError as e:
    print(f"❌ Error importing src.evaluate: {e}")

In [ ]:
data_dir = project_root / 'data'
processed_dir = data_dir / 'processed'

def inspect_dir(directory):
    if directory.exists():
        files = list(directory.glob('*.*'))
        if files:
            df_files = pd.DataFrame([
                {'File Name': f.name, 'Suffix': f.suffix, 'Size (MB)': f.stat().st_size / (1024*1024)} 
                for f in files
            ])
            display(df_files)
            return files
        else:
            print(f"Directory {directory} is empty.")
            return []
    else:
        print(f"Directory {directory} does not exist.")
        return []

print("Inspecting data/processed/:")
processed_files = inspect_dir(processed_dir)

print("\nInspecting data/:")
data_files = inspect_dir(data_dir)

# Determine correct test dataset
test_dataset_path = None

for f in processed_files + data_files:
    if f.name == 'processed_test.parquet' or f.name == 'test.csv' or f.name == 'X_test.csv':
        if 'test' in f.name:
            test_dataset_path = f
            break

if test_dataset_path:
    print(f"\n✅ Selected test dataset: {test_dataset_path}")
else:
    print("\n❌ No valid test dataset found!")

In [ ]:
X_test, y_test = None, None

if test_dataset_path:
    try:
        if test_dataset_path.suffix == '.parquet':
            test_df = pd.read_parquet(test_dataset_path)
        elif test_dataset_path.suffix == '.csv':
            test_df = pd.read_csv(test_dataset_path)
        else:
            raise ValueError(f"Unsupported file format: {test_dataset_path.suffix}")
        
        target_col = DEEP_LEARNING_TRAINING.get("target_column", "TARGET")
        
        if target_col in test_df.columns:
            X_test = test_df.drop(columns=[target_col])
            y_test = test_df[target_col]
            
            assert len(X_test) == len(y_test), "Mismatch in features and labels length!"
            assert set(y_test.unique()).issubset({0, 1}), "y_test must be binary (0 and 1)!"
            
            print(f"✅ Successfully loaded {test_dataset_path.name}")
            print(f"X_test shape: {X_test.shape}")
            print(f"y_test shape: {y_test.shape}")
            print(f"Number of features: {X_test.shape[1]}")
            print(f"Missing values in X_test: {X_test.isnull().sum().sum()}")
            print(f"TARGET distribution:\n{y_test.value_counts(normalize=True)}")
        else:
            print(f"❌ Target column '{target_col}' not found in dataframe!")
    except Exception as e:
        print(f"❌ Error loading test data: {e}")
else:
    print("❌ Cannot load test data because no valid path was found.")

In [ ]:
available_model_paths = {}
unavailable_models = {}

print("Model Discovery:\n" + "-"*40)
for name, filename in MODEL_NAMES.items():
    path = MODEL_DIR / filename
    if path.exists():
        available_model_paths[name] = path
        print(f"✅ {name}: Found at {path.name}")
    else:
        unavailable_models[name] = path
        print(f"❌ {name}: Not Found ({path.name})")

In [ ]:
import joblib

available_models = {}

# We create an adapter for Keras models so they work seamlessly with evaluate_model()
class KerasModelAdapter:
    def __init__(self, keras_model):
        self.model = keras_model
        
    def predict_proba(self, X):
        return self.model.predict(X, verbose=0)
        
    def predict(self, X):
        prob = self.predict_proba(X)
        return (prob >= 0.5).astype(int)

for name, path in available_model_paths.items():
    try:
        if path.suffix in ['.keras', '.h5']:
            import tensorflow as tf
            # Load and wrap in adapter
            model = KerasModelAdapter(tf.keras.models.load_model(path))
        else:
            model = joblib.load(path)
            
        available_models[name] = model
        print(f"✅ Successfully loaded {name}")
    except Exception as e:
        print(f"❌ Error loading {name}: {e}")

print(f"\nLoaded {len(available_models)} models successfully.")

In [ ]:
def get_positive_class_probability(model, X):
    if hasattr(model, 'predict_proba'):
        prob = model.predict_proba(X)
        if prob.ndim == 2:
            return prob[:, 1]
        return prob.flatten()
    elif hasattr(model, 'predict'):
        prob = model.predict(X, verbose=0) if 'keras' in str(type(model)).lower() else model.predict(X)
        return prob.flatten()
    else:
        raise ValueError("Model has neither predict_proba nor predict method.")

def get_class_prediction(model, X, threshold=0.5):
    prob = get_positive_class_probability(model, X)
    return (prob >= threshold).astype(int)

print("✅ INTERNAL prediction helpers created successfully.")

In [ ]:
evaluation_results = {}

if X_test is not None and y_test is not None:
    for name, model in available_models.items():
        try:
            # Using the evaluate_model API which internally calls predict_proba
            metrics = evaluate_model(model, X_test, y_test, threshold=0.5, plot=False)
            evaluation_results[name] = metrics
            print(f"✅ Evaluated {name}")
        except Exception as e:
            print(f"❌ Error evaluating {name}: {e}")
            # Fallback evaluation if evaluate_model fails
            try:
                print(f"Attempting fallback metric calculation for {name}...")
                from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, recall_score, precision_score
                prob = get_positive_class_probability(model, X_test)
                pred = get_class_prediction(model, X_test)
                evaluation_results[name] = {
                    "ROC AUC": roc_auc_score(y_test, prob),
                    "Average Precision": average_precision_score(y_test, prob),
                    "F1 Score": f1_score(y_test, pred),
                    "Recall": recall_score(y_test, pred),
                    "Precision": precision_score(y_test, pred)
                }
                print(f"✅ Fallback evaluation succeeded for {name}")
            except Exception as inner_e:
                print(f"❌ Fallback evaluation failed for {name}: {inner_e}")
else:
    print("Test data not available. Skipping evaluation.")

In [ ]:
if evaluation_results:
    comparison_df = pd.DataFrame.from_dict(evaluation_results, orient='index')
    comparison_df.index.name = 'Model'
    
    expected_cols = [
        'Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC', 
        'Average Precision', 'Balanced Accuracy', 'Matthews Corrcoef', 
        'Cohen Kappa', 'Log Loss', 'Brier Score Loss'
    ]
    
    # Ensure all expected columns exist, fill missing with NaN
    for col in expected_cols:
        if col not in comparison_df.columns:
            comparison_df[col] = np.nan
            
    if 'ROC AUC' in comparison_df.columns:
        comparison_df = comparison_df.sort_values(by='ROC AUC', ascending=False)
        
    display(comparison_df[expected_cols])
else:
    print("No evaluation results to display.")
    comparison_df = pd.DataFrame()

In [ ]:
business_results = {}

if X_test is not None and y_test is not None:
    for name, model in available_models.items():
        try:
            pred = get_class_prediction(model, X_test, threshold=0.5)
            metrics = business_metrics(y_test, pred)
            business_results[name] = metrics
        except Exception as e:
            print(f"❌ Error calculating business metrics for {name}: {e}")
            
    if business_results:
        business_df = pd.DataFrame.from_dict(business_results, orient='index')
        business_df.index.name = 'Model'
        display(business_df)
    else:
        print("No business metrics calculated.")
        business_df = pd.DataFrame()
else:
    print("Test data not available.")

In [ ]:
if not comparison_df.empty:
    try:
        best_roc = comparison_df['ROC AUC'].idxmax() if 'ROC AUC' in comparison_df.columns else "N/A"
        best_ap = comparison_df['Average Precision'].idxmax() if 'Average Precision' in comparison_df.columns else "N/A"
        best_recall = comparison_df['Recall'].idxmax() if 'Recall' in comparison_df.columns else "N/A"
        best_f1 = comparison_df['F1 Score'].idxmax() if 'F1 Score' in comparison_df.columns else "N/A"
        
        print("====== MODEL RANKING ======")
        print(f"Best ROC-AUC model:         {best_roc}")
        print(f"Best Average Precision:     {best_ap}")
        print(f"Best Recall model:          {best_recall}")
        print(f"Best F1 model:              {best_f1}")
        print("===========================")
    except Exception as e:
        print(f"Could not calculate ranking: {e}")
else:
    print("No data available for ranking.")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay

if available_models and X_test is not None and y_test is not None:
    fig, ax = plt.subplots(figsize=(10, 8))
    
    for name, model in available_models.items():
        try:
            prob = get_positive_class_probability(model, X_test)
            RocCurveDisplay.from_predictions(y_test, prob, name=name, ax=ax)
        except Exception as e:
            print(f"❌ Could not plot ROC curve for {name}: {e}")
            
    ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random Guess')
    ax.set_title('ROC Curve Comparison')
    ax.legend(loc='lower right')
    ax.grid(alpha=0.3)
    plt.show()
else:
    print("Models or data not available for ROC curve plotting.")

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay

if available_models and X_test is not None and y_test is not None:
    fig, ax = plt.subplots(figsize=(10, 8))
    
    for name, model in available_models.items():
        try:
            prob = get_positive_class_probability(model, X_test)
            PrecisionRecallDisplay.from_predictions(y_test, prob, name=name, ax=ax)
        except Exception as e:
            print(f"❌ Could not plot PR curve for {name}: {e}")
            
    baseline = y_test.mean()
    ax.axhline(baseline, linestyle='--', color='gray', label=f'Baseline (Prevalence: {baseline:.3f})')
    
    ax.set_title('Precision-Recall Curve Comparison')
    ax.legend(loc='best')
    ax.grid(alpha=0.3)
    plt.show()
else:
    print("Models or data not available for PR curve plotting.")

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

if available_models and X_test is not None and y_test is not None:
    n_models = len(available_models)
    cols = min(3, n_models)
    rows = (n_models + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 4))
    if n_models == 1:
        axes = np.array([axes])
    axes = axes.flatten()
    
    for idx, (name, model) in enumerate(available_models.items()):
        ax = axes[idx]
        try:
            pred = get_class_prediction(model, X_test, threshold=0.5)
            ConfusionMatrixDisplay.from_predictions(
                y_test, pred, 
                cmap='Blues', 
                ax=ax,
                colorbar=False
            )
            ax.set_title(f'Confusion Matrix: {name}')
        except Exception as e:
            ax.set_title(f'{name} - Error')
            ax.text(0.5, 0.5, 'Failed', ha='center', va='center')
            print(f"Error plotting confusion matrix for {name}: {e}")
            
    for idx in range(n_models, len(axes)):
        axes[idx].set_visible(False)
        
    plt.tight_layout()
    plt.show()
else:
    print("Models or data not available for Confusion Matrices.")

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

threshold_df = pd.DataFrame()

if not comparison_df.empty and X_test is not None and y_test is not None:
    champion_name = comparison_df['ROC AUC'].idxmax() if 'ROC AUC' in comparison_df.columns else list(available_models.keys())[0]
    best_model = available_models.get(champion_name)
    
    if best_model:
        print(f"Performing threshold analysis on champion model: {champion_name}")
        thresholds = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]
        results = []
        
        try:
            prob = get_positive_class_probability(best_model, X_test)
                
            for thresh in thresholds:
                pred = (prob >= thresh).astype(int)
                
                prec = precision_score(y_test, pred, zero_division=0)
                rec = recall_score(y_test, pred, zero_division=0)
                f1 = f1_score(y_test, pred, zero_division=0)
                
                res_dict = {
                    'Threshold': thresh,
                    'Precision': prec,
                    'Recall': rec,
                    'F1 Score': f1,
                }
                
                try:
                    bm = business_metrics(y_test, pred)
                    res_dict.update(bm)
                except Exception as bm_err:
                    pass
                
                results.append(res_dict)
                
            threshold_df = pd.DataFrame(results).set_index('Threshold')
            display(threshold_df)
            
        except Exception as e:
            print(f"❌ Error during threshold analysis: {e}")
    else:
        print("Best model could not be retrieved for threshold analysis.")
else:
    print("Data not available for threshold analysis.")

In [ ]:
if not threshold_df.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    
    ax.plot(threshold_df.index, threshold_df['Precision'], marker='o', label='Precision')
    ax.plot(threshold_df.index, threshold_df['Recall'], marker='s', label='Recall')
    ax.plot(threshold_df.index, threshold_df['F1 Score'], marker='^', label='F1 Score')
    
    ax.set_title(f'Threshold Analysis for {champion_name}')
    ax.set_xlabel('Decision Threshold')
    ax.set_ylabel('Score')
    ax.grid(alpha=0.3)
    ax.legend()
    plt.show()
else:
    print("No threshold data available for plotting.")

In [ ]:
if not comparison_df.empty and len(available_models) > 0:
    champion_name = comparison_df['ROC AUC'].idxmax() if 'ROC AUC' in comparison_df.columns else list(available_models.keys())[0]
    
    if 'ROC AUC' in comparison_df.columns:
        champion_score = comparison_df.loc[champion_name, 'ROC AUC']
    else:
        champion_score = "N/A"
        
    champion_path = available_model_paths.get(champion_name, "Unknown Path")
        
    print("========================================")
    print("         CHAMPION MODEL DETECTED        ")
    print("========================================")
    print(f"Champion Model:       {champion_name}")
    print(f"Selection Metric:     ROC AUC")
    print(f"Selection Score:      {champion_score}")
    print(f"Model Artifact Path:  {champion_path}")
    print("========================================")
else:
    print("No models were available to select a champion.")

In [ ]:
REPORT_DIR.mkdir(exist_ok=True)
saved_files = []

if not comparison_df.empty:
    comp_path = REPORT_DIR / 'model_comparison.csv'
    comparison_df.to_csv(comp_path)
    saved_files.append(comp_path)
    
if 'business_df' in locals() and not business_df.empty:
    bus_path = REPORT_DIR / 'business_metrics.csv'
    business_df.to_csv(bus_path)
    saved_files.append(bus_path)
    
if not threshold_df.empty:
    thresh_path = REPORT_DIR / 'threshold_analysis.csv'
    threshold_df.to_csv(thresh_path)
    saved_files.append(thresh_path)
    
if saved_files:
    print("✅ Successfully saved the following reports:")
    for f in saved_files:
        print(f" - {f}")
else:
    print("No data available to save.")

In [ ]:
from IPython.display import display, Markdown

if not comparison_df.empty:
    summary_md = f"""
### Executive Summary

* **Evaluated Models**: `{', '.join(available_models.keys())}`
* **Unavailable Models**: `{', '.join(unavailable_models.keys()) if unavailable_models else 'None'}`

**Champion Model:**
* **Model**: `{champion_name}`
* **ROC AUC**: `{champion_score}`
* **Best Average Precision**: `{best_ap}`
* **Best Recall**: `{best_recall}`
* **Best F1 Score**: `{best_f1}`

**Threshold Analysis Conclusion:**
The threshold analysis indicates a clear trade-off between Precision and Recall. Lowering the threshold increases the Recall (capturing more defaults) but drops Precision (increasing False Positives), which translates to more rejected loans for good customers. The optimal threshold should be chosen based on the business cost of a false positive vs a false negative.
"""
    display(Markdown(summary_md))
else:
    print("Summary cannot be generated due to lack of evaluation results.")